# ParametricNAM Training

Select a GPU runtime. Put this repository and the exported Capture `dataset/` folder on Drive. Keep all files inside the exported folder together. Pick a fresh output directory for each dataset, and reuse that directory to continue the same run.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import subprocess
import sys

PROJECT = Path('/content/drive/MyDrive/ParametricNAM-Training')
DATASET = Path('/content/drive/MyDrive/CaptureDataset')
OUTPUT = Path('/content/drive/MyDrive/ParametricNAM-runs/amp-v1')
TOTAL_STEPS = 20000
BATCH_SIZE = 12


In [ ]:
from importlib.metadata import PackageNotFoundError, version

constraints = Path('/content/pnam-torch-constraints.txt')
pins = []
for package in ('torch', 'torchaudio', 'torchvision'):
    try:
        pins.append(f'{package}=={version(package)}')
    except PackageNotFoundError:
        pass
constraints.write_text('\n'.join(pins) + '\n')
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-c', str(constraints),
    '-r', str(PROJECT / 'requirements-colab.txt')
], check=True)

import torch
assert torch.cuda.is_available(), 'Switch this notebook to a GPU runtime.'


In [ ]:
subprocess.run([
    sys.executable, '-u', str(PROJECT / 'run_training.py'),
    '--manifest', str(DATASET / 'manifest.json'),
    '--profile', str(DATASET / 'profile.json'),
    '--out', str(OUTPUT), '--scratch', '/content',
    '--steps', str(TOTAL_STEPS), '--batch-size', str(BATCH_SIZE), '--device', 'cuda',
], check=True)
print(OUTPUT / 'model.pnam')


For a short smoke run, set a total of 2 steps and a batch size of 1 in a separate output directory. Once the main run completes, download `model.pnam` from its output directory and open it in ParametricNAM.